In [ ]:
%%capture
!pip install unsloth
!pip install pillow==11.3.0

In [ ]:
# imports

from unsloth import FastVisionModel          # MUST be first
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch
import os
import json
from PIL import Image
from huggingface_hub import login, snapshot_download
import time

# this line came from 01 zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

In [ ]:
# one-time execute to download the model
path = snapshot_download(
    repo_id="unsloth/gemma-4-e4b-it",
    local_dir="/kaggle/working/gemma4-unsloth",
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")


In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/gemma4-unsloth",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Model successfully loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# attach LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=16, # ~36M params
    lora_alpha=32, # (usually 2*r) how strongly LoRA nudges the output
    # attention projection layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=False, # train text only for phase 1
)

# check if LoRA successfully attached
print(model.print_trainable_parameters())


In [ ]:
# load our civicinsight dataset
DATASET_JSON = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/dataset.marked.json"
KAGGLE_IMAGE_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/standardized/standardized"

# build the message
# Input: PIL image
#        aria_label
#        prompt
# Output: message built as prompt with inputs
def to_conversation(img, aria_label, prompt):
    return {
        "messages": [
            {
                "role": "user", # one role as user with image and prompt
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ],
            },
            {
                "role": "assistant", # one role as assistant on what to train on
                "content": [
                     {"type": "text", "text": aria_label},
                ]              
            }
        ]
    }
    
with open(DATASET_JSON) as f:
    dataset = json.load(f)

conversations = []
for record in dataset:
    # fix the path name from the dataset to match the Kaggle dataset dir format
    img_path = os.path.join(KAGGLE_IMAGE_DIR, os.path.basename(record["image"]))
    # convert to PIL
    img = Image.open(img_path)
    # convert to conversation, append to conversations.
    conversations.append(to_conversation(img, record["aria_label"], record["prompt"]))

# sanity check, always
print(f"Loaded {len(conversations)} conversations")
print(conversations[0]["messages"][0]["content"][0])  # should print the prompt text


In [ ]:
# epoch run test
trainer = SFTTrainer(
    model=model,                                          # the LoRA-wrapped Gemma 4
    tokenizer=tokenizer,                                  # handles text + image tokenization
    data_collator=UnslothVisionDataCollator(model, tokenizer),  # batches image+text together (vision-specific, replaces default)
    train_dataset=conversations,                          # full 50 examples; max_steps stops early
    args=SFTConfig(
        per_device_train_batch_size=1,                   # 1 example per forward pass (GPU memory constraint)
        gradient_accumulation_steps=4,                   # accumulate 4 steps before updating weights = effective batch of 4
        num_train_epochs=5,                              # run our standar 3-epoch test.
        learning_rate=2e-4,                              # standard LoRA starting LR
        output_dir="./test_output",                      # checkpoint save location (kaggle/working/)
        max_seq_length=2048,                             # max tokens per example (image + text combined)
        dataset_text_field="",                           # tells SFT not to look for a text column (we handle format ourselves)
        dataset_kwargs={"skip_prepare_dataset": True},   # skip HuggingFace auto-formatting (our format is already correct)
    )
)

print(f"GPU before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()
print("Training ran without crash!")
print(f"GPU after training: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
import time

# run inference on an image from training set already (this printed exactly the same label back again)
model.eval()                 # switch to eval mode
message = conversations[5]["messages"][:1]   # pick up an existing conversation

inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400)
elapsed = time.time() - start

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\n{'='*60}")
print(f"IMAGE: {message} | Time: {elapsed:.1f}s")
print(f"{'='*60}")
print(decoded)


In [ ]:
# run inference on a never seen image
KAGGLE_TEST_IMG_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-test"

img_path = os.path.join(KAGGLE_TEST_IMG_DIR, "browser-share.png")
img = Image.open(img_path)
prompt = "Generate an aria-label for this data visualization image."

message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]

inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

start = time.time()
outputs = model.generate(**inputs, max_new_tokens=2048)
elapsed = time.time() - start

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(decoded)
